# Reproducing Even-Chen et al. (2019) from dandiset 000121

Paper. Even-Chen N, Sheffer B, Vyas S, Ryu SI, Shenoy KV (2019). Structure and variability of
delay activity in premotor cortex. *PLOS Computational Biology* 15(2), e1006808.
[doi:10.1371/journal.pcbi.1006808](https://doi.org/10.1371/journal.pcbi.1006808). This attempt
used the full text, which is open access.

Dandiset. [DANDI:000121](https://dandiarchive.org/dandiset/000121/0.220124.2156), version
0.220124.2156. The paper cites no DANDI version (it predates the dandiset; its data statement
points to a Stanford zip file that now returns HTTP 403). This is the most recent published
version. The earlier published version, 0.210815.0703, holds the same 12 files (same blobs and
sizes) and differs only in metadata; the draft holds no files (see the data check).

What the dandiset holds. 12 sessions of delayed center-out reaching with two Utah arrays: all
7 sessions of monkey R and 5 of the 16 of monkey J (ring 1 of 4, vertical 2 of 4, 3-rings 2 of 4,
horizontal none). Each file has a trials table, hand, cursor and eye positions, threshold
crossings of the 192 electrodes as units, and LFP.

Code read. The paper names no analysis code. The conversion code,
[catalystneuro/shenoy-lab-to-nwb](https://github.com/catalystneuro/shenoy-lab-to-nwb) at commit
319acb472372c9e4359ebacf13c8df3652a822b1 (`center_out_task/`), was read. It builds the units from
the rig's 1 ms threshold-crossing rasters of the two arrays (units 0 to 95 from `spikeRaster`, 96
to 191 from `spikeRaster2`), takes `go_cue_time` from the rig's `timeTargetOn` and
`target_shown_time` as that minus `delayTime`, assigns the units' `electrode_group` by the order of
a dictionary of groups, and stores positions with `conversion` NaN. It settles no analysis
choice.

Outside inputs. None.

No claim uses data from outside the dandiset. Random numbers come from generators seeded per
section (the seeds are named at the top of each section).

| Claim | Status |
|---|---|
| session-configurations | tested |
| r-trial-counts | tested |
| j-sessions-complete | blocked: 11 of J's 16 sessions are not in the dandiset |
| delays-and-times | tested |
| channels-task-correlated | tested |
| pca-variance | tested (J on 5 of 16 sessions) |
| spatial-plane-variance | tested (J on 5 of 16 sessions, without horizontal) |
| plane-timecourse | tested |
| onoff-predicts-kinematics | tested |
| speed-axis-within-distance | tested (J on its 2 vertical sessions) |
| distance-after-speed | tested |
| speed-distance-projection | tested |
| single-trial-variance | tested |
| ellipse-orientation | tested |
| on-greater-than-off | tested |
| relative-deviation-decreases | tested |
| svm-ring-and-distance | tested (J ring on 1 of 4 sessions) |
| svm-3rings-by-ring | tested |
| arc-error-distance-independent | tested |
| distance-error-and-plane-projection | tested |


In [ ]:
import json
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

import premotor

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = premotor.load_all()
print(f"loaded {len(sessions)} sessions from dandiset {premotor.DANDISET_ID} version "
      f"{premotor.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
API = "https://api.dandiarchive.org/api/dandisets/000121"


def get(url):
    with urllib.request.urlopen(url) as r:
        return json.load(r)


versions = [v["version"] for v in get(f"{API}/versions/")["results"]]
print("Versions:", versions)
listing = {}
for v in versions:
    assets = get(f"{API}/versions/{v}/assets/?page_size=100")["results"]
    listing[v] = sorted((a["path"], a.get("blob"), a["size"]) for a in assets)
    print(f"  {v}: {len(assets)} assets, {sum(a['size'] for a in assets) / 1e9:.1f} GB")
published = [v for v in versions if v != "draft"]
print("Published versions hold the same files (path, blob, size):",
      listing[published[0]] == listing[published[1]])
meta = get(f"{API}/versions/{premotor.DANDISET_VERSION}/")
summary = meta["assetsSummary"]
print("assetsSummary of the pinned version: numberOfFiles", summary["numberOfFiles"],
      "; numberOfBytes", f"{summary['numberOfBytes'] / 1e9:.1f} GB",
      "; numberOfSubjects", summary["numberOfSubjects"])
print("Description:", meta["description"])

Both published versions list the same 12 files with the same blobs; they differ only in metadata
(schema version, contributor roles). The draft holds no assets. The archive's summary of the
pinned version gives 15 files and 187.5 GB, while the version holds 12 files and 39.5 GB.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    co = t[t.center_out]
    delayed = co[co.successful & (co.delay >= premotor.MIN_DELAY_S)].delay
    rows.append({
        "monkey": s["monkey"], "date": s["date"], "trials": len(t), "successful": int(t.successful.sum()),
        "center_out": len(co), "succ_center_out": int(co.successful.sum()),
        "succ_co_no_delay": int((co.successful & (co.delay < premotor.MIN_DELAY_S)).sum()),
        "delay_min_ms": round(1000 * delayed.min()), "delay_max_ms": round(1000 * delayed.max()),
        "off_layout": int((co.successful & ~co.in_layout).sum()), "analysed": int(t.analysed.sum()),
        "task_id": sorted(t.task_id.unique().tolist()), "version_id": sorted(t.version_id.unique().tolist()),
        "targets": co[["tx", "ty"]].round(1).drop_duplicates().shape[0],
        "hand_samples": s["n_hand_samples"], "hand_dt_ms": round(1000 * s["hand_dt_median"], 3),
        "recorded_s": round(s["recorded_s"]),
        "cursor_minus_hand_mm_p50_p99_max": tuple(np.round(s["meta"]["cursor_minus_hand_mm"], 2))})
overview = pd.DataFrame(rows)
print(overview.to_string(index=False))
print("\nDelays of successful center-out trials below", premotor.MIN_DELAY_S, "s, by value (ms):")
for m in "JR":
    d = pd.concat([s["trials"] for s in sessions if s["monkey"] == m])
    d = d[d.successful & d.center_out & (d.delay < premotor.MIN_DELAY_S)].delay
    print(" ", m, (1000 * d).round().value_counts().sort_index().to_dict())
print("\nTarget positions (stated unit none; values):",
      {s["date"]: (round(s["trials"].tx.min()), round(s["trials"].tx.max()), round(s["trials"].ty.min()),
                   round(s["trials"].ty.max())) for s in sessions})

`task_id` is 2 and `version_id` one value per monkey in every trial, so the files do not say which
target configuration a session used; it is inferred from the target positions (see
`session-configurations`). Target positions run to 120, the 12 cm of the outer ring, so they are
in mm. The hand and cursor positions are stated in meters with a `conversion` of NaN, but take
the same mm values (the target of 80 is reached at about 80, see the clock check below).
Cursor and hand agree to within 1.9 to 2.8 mm in 99% of samples in 11 sessions; in R's
2017-01-19 they differ by more than 1 m in over 1% of samples (up to 1903 mm, beyond any
screen position). Only the hand is used here. Hand samples are 1 ms apart and exist only within trials.
Delays of delayed trials run from 300 to 700 ms for J and from 400 to 1000 ms for R. Trials
without a delay have delays of 0 or 1 ms; J has them in every session, R only 26, all on
2017-01-19.

In [ ]:
rows = []
for s in sessions:
    u = s["units"]
    pmd, lab = u.iloc[premotor.PMD_UNITS], u.iloc[premotor.FILE_PMD_UNITS]
    rows.append({
        "monkey": s["monkey"], "date": s["date"],
        "units_0_95_label": "/".join(sorted(set(lab.group_label))),
        "units_0_95_electrode_rows": f"{lab.electrode.min()}-{lab.electrode.max()} ({'/'.join(sorted(set(lab.electrode_group)))})",
        "units_96_191_label": "/".join(sorted(set(pmd.group_label))),
        "units_96_191_electrode_rows": f"{pmd.electrode.min()}-{pmd.electrode.max()} ({'/'.join(sorted(set(pmd.electrode_group)))})",
        "lfp": s["meta"]["lfp_electrodes"],
        "live_0_95": int((lab.n_spikes > 0).sum()), "live_96_191": int((pmd.n_spikes > 0).sum()),
        "rate_0_95": round(lab.rate.median(), 1), "rate_96_191": round(pmd.rate.median(), 1),
        "short_span_96_191": int((pmd["last"] - pmd["first"] < 0.9 * s["recorded_s"]).sum()),
        "spikes_in_trials_96_191": round(pmd.in_trials.sum() / pmd.n_spikes.sum(), 4),
        "trial_s/recorded_s/span_s": (round(s["trial_s"]), round(s["recorded_s"]), round(s["span_s"])),
        "units_with_time_decreasing": [int(i) for i in np.flatnonzero(u.time_decreases)],
        "unit0_span_s": (round(u["first"][0], 1), round(u["last"][0], 1)),
        "unit1_n_spikes_vs_median": (int(u.n_spikes[1]), int(u.n_spikes.median()))})
print(pd.DataFrame(rows).to_string(index=False))
print("\nElectrode groups: 1 =", "M1", "; 2 = PMd (from general/extracellular_ephys)")

Units 0 to 95 carry `electrode_group` 2 (PMd), but their `electrodes` column points to electrode
rows 0 to 95, which belong to group 1 (M1); units 96 to 191 are labelled M1 and point to rows of
group 2 (PMd), the rows the PMd LFP uses. The conversion code assigns `electrode_group` by the
order of a dictionary of groups, not from the recording. The next cell asks which array carries
the delay-period target signal.

In every J file unit 1's spike times decrease once, from the end of the session back to its
start, unit 1 holds two to five times the median number of spikes, and unit 0 stops long before
the session ends: the boundary between units 0 and 1 in `spike_times_index` is misplaced, and
unit 1 holds the rest of unit 0's train followed by its own. Both are on the array of units 0 to
95, not the one used here as PMd. The loader sorts any unsorted train before counting.

Spikes, like hand samples, exist only within trials: 99.93 to 99.95% of the PMd spikes of every
session fall inside a trial interval, although each file's span exceeds its summed trial time by
484 to 1963 s (the gaps between trials hold almost no spikes). The recorded time of the
hand samples equals the summed trial time to within 7 s, so the day's mean rate, the spike count
over the recorded time, is the rate over trial time. The per-session median PMd rates still vary
widely, from 4.1 to 12.0 Hz in J and 4.6 to 47.4 Hz in R.

In [ ]:
from sklearn.model_selection import StratifiedKFold  # noqa: E402

ARRAY_SEED = 1
rows = []
for s in sessions:
    t = s["trials"][s["trials"].analysed]
    y = (np.round(np.degrees(np.arctan2(t.ty, t.tx))) * 1000 + np.round(np.hypot(t.tx, t.ty))).values
    keep = pd.Series(y).map(pd.Series(y).value_counts()) >= 5
    for name, units in (("units 0-95", premotor.FILE_PMD_UNITS), ("units 96-191", premotor.PMD_UNITS)):
        row = {"monkey": s["monkey"], "date": s["date"], "array": name}
        for window, counts in (("delay", s["delay_counts"]), ("move", s["move_counts"])):
            X = counts[:, units].astype(float)
            Z = np.c_[np.ones(len(t)), t.tx, t.ty]
            n_sig = 0
            for ch in range(X.shape[1]):
                if X[:, ch].std() == 0:
                    continue
                beta = np.linalg.lstsq(Z, X[:, ch], rcond=None)[0]
                rss = ((X[:, ch] - Z @ beta) ** 2).sum()
                tss = ((X[:, ch] - X[:, ch].mean()) ** 2).sum()
                F = ((tss - rss) / 2) / (rss / (len(X) - 3))
                n_sig += stats.f.sf(F, 2, len(X) - 3) < 0.05 / 96
            _, acc = premotor.svm_cv(X[keep.values], y[keep.values], ARRAY_SEED, folds=5)
            row[f"{window}_sig_channels"] = n_sig
            row[f"{window}_accuracy"] = round(acc.mean(), 3)
        row["chance"] = round(1 / len(np.unique(y[keep.values])), 3)
        rows.append(row)
arrays = pd.DataFrame(rows)
print("Channels whose count regresses on target x, y (F-test, p < 0.05/96), and 5-fold SVM accuracy "
      "for the target, in the 200 ms before the go cue (delay) and 200 to 400 ms after it (move):")
print(arrays.to_string(index=False))
ratio = arrays.assign(r=arrays.delay_accuracy / arrays.move_accuracy).pivot_table(
    index="monkey", columns="array", values=["delay_sig_channels", "delay_accuracy", "r"], aggfunc="median")
print("\nMedians over sessions (r = delay accuracy / move accuracy):")
print(ratio.round(3).to_string())

In R, units 0 to 95 carry almost no target information (at most 6 significant channels in the
delay window and 16 after the go cue, accuracy at most 0.03 above chance) and 29 of them never
fire: this fits the M1 array the paper calls too poor to use. Units 96 to 191 carry it in every
session (30 to 70 significant channels in the delay window). In J both arrays carry it; units 96
to 191 have the higher delay-window accuracy in all five sessions, more significant channels in
four of five, and a higher ratio of delay to movement accuracy (median 0.67 against 0.41), as
expected of PMd against M1. The data therefore agree
with the `electrodes` column and the LFP, not with the units' `electrode_group`: the analysis
takes units 96 to 191 as PMd. This is consistent with the data, not proven by them.

In [ ]:
CLOCK_SEED = 2
rows = []
for s in sessions:
    t = s["trials"][s["trials"].analysed]
    y = (np.round(np.degrees(np.arctan2(t.ty, t.tx))) * 1000 + np.round(np.hypot(t.tx, t.ty))).values
    keep = (pd.Series(y).map(pd.Series(y).value_counts()) >= 5).values
    tc = s["tc_counts"].astype(float)
    centers = premotor.TC_START_S + premotor.BIN_S * (np.arange(tc.shape[2]) + 0.5)
    windows = {"-200 to 0 ms (before target)": (centers < 0),
               "100 to 300 ms after target": (centers > 0.1) & (centers < 0.3)}
    row = {"monkey": s["monkey"], "date": s["date"], "chance": round(1 / len(np.unique(y[keep])), 3)}
    for name, w in windows.items():
        X = tc[:, :, w].sum(axis=2)
        row[name] = round(premotor.svm_cv(X[keep], y[keep], CLOCK_SEED, folds=5)[1].mean(), 3)
    X = s["delay_counts"][:, premotor.PMD_UNITS].astype(float)
    row["delay window"] = round(premotor.svm_cv(X[keep], y[keep], CLOCK_SEED, folds=5)[1].mean(), 3)
    shifted = np.roll(y, 1)
    keep2 = (pd.Series(shifted).map(pd.Series(shifted).value_counts()) >= 5).values
    row["delay window, labels of the previous trial"] = round(
        premotor.svm_cv(X[keep2], shifted[keep2], CLOCK_SEED, folds=5)[1].mean(), 3)
    k = s["trials"][s["trials"].successful & s["trials"].center_out & s["trials"].max_speed.notna()]
    row["onset before go cue (fraction)"] = round(float((k.reaction_time < 0).mean()), 3)
    row["median reaction time (ms)"] = round(1000 * k.reaction_time.median())
    row["hand at max speed, median along target (mm)"] = round(float(np.median(
        (k.x_at_max * k.tx + k.y_at_max * k.ty) / np.hypot(k.tx, k.ty))))
    rows.append(row)
clock = pd.DataFrame(rows)
print("5-fold SVM accuracy for the target from PMd counts, by window, and hand timing:")
print(clock.to_string(index=False))

In every session the target cannot be decoded in the 200 ms before target onset (accuracy at
chance), and can be 100 to 300 ms after it and in the 200 ms before the go cue; with each trial
given the previous trial's target, accuracy in the delay window falls to near chance (at most
0.023 above it, against 0.067 to 0.207 above it with the true targets). The spike times
therefore share the clock of the trial table, to the resolution of these windows. Median
reaction times are 306 to 379 ms in every session, and the hand at maximum speed lies along the
target direction (median 31 to 42 mm along it), so the hand shares the clock too. Movement onset
(10% of peak speed) precedes the go cue in 7 to 11% of trials in R's four horizontal and
vertical sessions and in at most 5% elsewhere; these trials are kept.

In [ ]:
rows = []
for m in "JR":
    ss = [s for s in sessions if s["monkey"] == m]
    rates = np.array([s["units"].rate.values for s in ss])
    counts = [len(s["trials"]) for s in ss]
    c = np.corrcoef(rates)
    np.fill_diagonal(c, np.nan)
    print(f"{m}: trials per session {counts}; hand samples {[s['n_hand_samples'] for s in ss]}")
    print(f"   highest correlation between sessions of the 192 channel rates: {np.nanmax(c):.3f}; "
          f"identical trial counts: {len(set(counts)) < len(counts)}")
    first = [tuple(np.round(s["trials"][["go_cue_time", "tx", "ty"]].dropna().iloc[:5].values.ravel(), 3)) for s in ss]
    print("   identical first five trials in two sessions:", len(set(first)) < len(first))

No two sessions share trial or sample counts, their first trials, or their channel rates, so no
recording was deposited twice. The analysis combines sessions of a monkey and treats each
channel (electrode) as the same across sessions, as the paper does; this is an assumption the
data cannot test, since threshold crossings carry no identity. Results that rest on it are also
given per session where cheap.

In [ ]:
for m in "JR":
    d = pd.concat([s["trials"] for s in sessions if s["monkey"] == m])
    k = d[d.successful & d.center_out]
    print(f"{m}: successful center-out trials {len(k)}; with a hand glitch (|position| > "
          f"{premotor.HAND_LIMIT_MM:.0f} mm) {int(k.glitch.sum())}; without kinematics {int(k.max_speed.isna().sum())}")
    print(f"   max speed (mm/s) range {k.max_speed.min():.0f} to {k.max_speed.max():.0f}, median {k.max_speed.median():.0f}; "
          f"reaction time (ms) 5th to 95th percentile {1000 * k.reaction_time.quantile(0.05):.0f} to "
          f"{1000 * k.reaction_time.quantile(0.95):.0f}")
    a, _ = premotor.monkey_table(sessions, m)
    size = a.groupby("condition").size()
    print(f"   analysed trials {len(a)} in {len(size)} conditions; trials per condition {size.min()} to {size.max()}")

## session-configurations

> The sessions are those the Methods list for each target configuration; for monkey R, ring on
> 2017-01-24 and 25, horizontal on 2017-01-17 and 18, vertical on 2017-01-15 and 16, and 3-rings
> on 2017-01-19; for J, ring, vertical and 3-rings on the dates listed.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    co = t[t.center_out & t.successful]
    off = co[~co.in_layout]
    rows.append({"monkey": s["monkey"], "date": s["date"], "configuration in the file": s["config"],
                 "Methods": s["methods_config"], "match": s["config"] == s["methods_config"],
                 "radii (mm)": sorted(np.unique(np.round(np.hypot(co.tx, co.ty))).astype(int).tolist()),
                 "targets": co[["tx", "ty"]].round(1).drop_duplicates().shape[0],
                 "trials off the layout": len(off),
                 "their targets": sorted(set(zip(off.tx.round(1), off.ty.round(1))))})
configs = pd.DataFrame(rows)
print(configs.to_string(index=False))
print("\nSessions matching the Methods:", int(configs.match.sum()), "of", len(configs))
swapped = configs[~configs.match]
print("Mismatches that are horizontal and vertical exchanged:",
      int((swapped["configuration in the file"].isin(["horizontal", "vertical"])
           & swapped.Methods.isin(["horizontal", "vertical"])).sum()), "of", len(swapped))
for m in "JR":
    listed = sorted(premotor.METHODS_CONFIG[m])
    present = sorted(s["date"] for s in sessions if s["monkey"] == m)
    print(f"\n{m}: the Methods list {len(listed)} sessions, the dandiset holds {len(present)}; missing:",
          [d for d in listed if d not in present])

**Result: partially reproduced.** 8 of 12 sessions have the configuration the Methods give for
their date. The other 4 are R's: 2017-01-15 and 16 hold horizontal targets where the Methods say
vertical, and 2017-01-17 and 18 hold vertical targets where the Methods say horizontal, so all
four mismatches are horizontal and vertical exchanged. 2017-01-17 also holds 69 successful trials
to six ring targets outside its layout. The dandiset holds all 7 of R's sessions and 5 of J's 16;
the 11 missing include all four of J's horizontal sessions.

## r-trial-counts

> Monkey R performed 3577 ring trials over two sessions, 3699 horizontal over two, 4040 vertical
> over two, and 912 3-rings trials in one session.

In [ ]:
R_EXPECTED = {"ring": 3577, "horizontal": 3699, "vertical": 4040, "3-rings": 912}
rows = []
for s in sessions:
    if s["monkey"] != "R":
        continue
    t = s["trials"]
    co = t[t.center_out]
    rows.append({"date": s["date"], "Methods": s["methods_config"], "file": s["config"],
                 "(a) successful, delayed": int((co.successful & (co.delay >= premotor.MIN_DELAY_S)).sum()),
                 "(b) delayed, any outcome": int((co.delay >= premotor.MIN_DELAY_S).sum()),
                 "(c) successful, with no-delay": int(co.successful.sum()),
                 "(b) also without a go cue": int((~(co.delay < premotor.MIN_DELAY_S)).sum()),
                 "(a) in the layout only": int((co.successful & (co.delay >= premotor.MIN_DELAY_S) & co.in_layout).sum())})
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
readings = ["(a) successful, delayed", "(b) delayed, any outcome", "(c) successful, with no-delay",
            "(b) also without a go cue", "(a) in the layout only"]
for grouping in ("Methods", "file"):
    g = counts.groupby(grouping)[readings].sum().reindex(list(R_EXPECTED))
    g["paper"] = pd.Series(R_EXPECTED)
    print(f"\nGrouped by the configuration of the {grouping}:")
    print(g.to_string())
    for r in readings:
        ok = (g[r] - g.paper).abs() <= 0.1 * g.paper
        print(f"   {r}: within 10% for {int(ok.sum())} of 4 ({', '.join(g.index[ok])})")

**Result: partially reproduced.** Grouped by the Methods' dates, two of the four counts are
within 10% under readings (a) to (c): ring 3584 against 3577 and vertical 3842 against 4040 under
reading (a). Horizontal gives 4156 against 3699 and 3-rings 1340 against 912. Reading (b) drops
failed trials without a go cue (delay undefined); counting them as well gives no match (ring
4568, horizontal 5250, vertical 5033, 3-rings 1671). Grouped by the geometry in the files, three
of four are within 10% (horizontal 3842 against 3699, vertical 4156 against 4040), which fits the
exchange of R's horizontal and vertical dates found above. The 3-rings session holds 1340
successful delayed trials where the paper used 912.

## delays-and-times

> Delays were uniform from 300 to 700 ms for J and 400 to 900 ms for R, 10% of trials had no
> delay; average reaction time was 346 ms (J) and 311 ms (R), average reach time 582 ms (J) and
> 482 ms (R), and reaction time was negatively correlated with delay (p < 1e-4 for both).

In [ ]:
for m in "JR":
    d = pd.concat([s["trials"].assign(date=s["date"]) for s in sessions if s["monkey"] == m])
    k = d[d.successful & d.center_out]
    delayed = k[k.delay >= premotor.MIN_DELAY_S]
    kk = delayed[delayed.max_speed.notna() & delayed.onset.notna()]
    rt = 1000 * kk.reaction_time
    reach_a = 1000 * (kk.first_acquire - kk.onset)
    reach_b = 1000 * (kk.first_acquire - kk.go_cue_time)
    r, p = stats.pearsonr(kk.delay, kk.reaction_time)
    print(f"{m}: delays of delayed trials {1000 * delayed.delay.min():.0f} to {1000 * delayed.delay.max():.0f} ms; "
          f"deciles {np.round(1000 * delayed.delay.quantile(np.linspace(0, 1, 11))).astype(int).tolist()}")
    print(f"   no-delay fraction of successful center-out trials {np.mean(k.delay < premotor.MIN_DELAY_S):.3f} "
          f"({int((k.delay < premotor.MIN_DELAY_S).sum())} of {len(k)})")
    print(f"   reaction time mean {rt.mean():.0f} ms (median {rt.median():.0f}); reach time (a) acquisition - onset "
          f"mean {reach_a.mean():.0f} ms (median {reach_a.median():.0f}); (b) acquisition - go cue mean "
          f"{reach_b.mean():.0f} ms (median {reach_b.median():.0f}); trials {len(kk)}, without kinematics "
          f"{int(delayed.max_speed.isna().sum())}, without an onset {int((delayed.max_speed.notna() & delayed.onset.isna()).sum())}, "
          f"without acquisition {int(kk.first_acquire.isna().sum())}")
    print(f"   reaction time against delay: r = {r:.3f}, p = {p:.1e}")
    rt_trim = kk[(kk.reaction_time > 0.1) & (kk.reaction_time < 0.8)]
    r2, p2 = stats.pearsonr(rt_trim.delay, rt_trim.reaction_time)
    print(f"   variant, reaction times 100 to 800 ms only ({len(rt_trim)} trials): mean {1000 * rt_trim.reaction_time.mean():.0f} ms, "
          f"r = {r2:.3f}, p = {p2:.1e}")
    v = kk[kk.onset_after_go.notna()]
    rt_go = 1000 * (v.onset_after_go - v.go_cue_time)
    rg, pg = stats.pearsonr(v.delay, v.onset_after_go - v.go_cue_time)
    print(f"   onset before the go cue in {np.mean(kk.reaction_time < 0):.3f} of trials; variant, onset searched only after "
          f"the go cue: reaction time mean {rt_go.mean():.0f} ms (median {rt_go.median():.0f}), reach time (b) unchanged, "
          f"trials without such an onset {len(kk) - len(v)} of {len(kk)}; r with delay {rg:.3f}, p = {pg:.1e}")
    for key, label in (("onset_05", "5%"), ("onset_20", "20%")):
        v = kk[kk[key].notna()]
        rv, pv = stats.pearsonr(v.delay, v[key] - v.go_cue_time)
        print(f"   onset at {label} of peak speed: reaction time mean {1000 * (v[key] - v.go_cue_time).mean():.0f} ms, "
              f"reach time (a) mean {1000 * (v.first_acquire - v[key]).mean():.0f} ms; r with delay {rv:.3f}, p = {pv:.1e}")
    per = kk.groupby("date").apply(lambda x: pd.Series({
        "rt_mean": 1000 * x.reaction_time.mean(), "reach_a": 1000 * (x.first_acquire - x.onset).mean(),
        "reach_b": 1000 * (x.first_acquire - x.go_cue_time).mean(),
        "r_delay": stats.pearsonr(x.delay, x.reaction_time)[0], "p": stats.pearsonr(x.delay, x.reaction_time)[1]}))
    per[["rt_mean", "reach_a", "reach_b"]] = per[["rt_mean", "reach_a", "reach_b"]].round(0)
    per["r_delay"] = per.r_delay.round(3)
    print(per.to_string(formatters={"p": "{:.1e}".format}))

**Result: partially reproduced.** J's delays run from 300 to 700 ms, inside the stated range;
R's run from 400 to 1000 ms, past the widened upper bound of 990 ms. Trials without a delay are
19.8% of J's successful center-out trials and 0.2% of R's (26 trials), not 10%. Mean reaction
times are 332 ms (J) and 305 ms (R), within tolerance of 346 and 311 ms, and reach times read as
acquisition minus go cue are 587 and 486 ms, within tolerance of 582 and 482 ms (read as
acquisition minus movement onset they are 255 and 181 ms). R's mean reaction time includes the
5.9% of its trials whose onset precedes the go cue (J 1.4%); R's median is 361 ms and its mean
over reaction times of 100 to 800 ms 362 ms, both outside 311 +- 31 ms. With onset searched only
after the go cue the means are 342 ms (J) and 340 ms (R), the latter just inside the bound of
342 ms, with 124 (J) and 759 (R) trials left without an onset. Reaction time falls with delay in
R (r = -0.264, p = 1.8e-205) but not in J (r = -0.013, p = 0.21); with onset at 5% of peak speed
J gives r = -0.034, p = 1.5e-03, and with onset searched after the go cue r = -0.029, p =
7.9e-03, both above 1e-4.

## channels-task-correlated

> Of the 96 PMd electrodes, 74 for J and 80 for R had activity significantly correlated with the
> x and y location of the target (p < .05, Bonferroni over 96).

In [ ]:
def significant_channels(t, X):
    Z = np.c_[np.ones(len(t)), t.tx, t.ty]
    p = np.ones(X.shape[1])
    for ch in range(X.shape[1]):
        if X[:, ch].std() == 0:
            continue
        beta = np.linalg.lstsq(Z, X[:, ch], rcond=None)[0]
        rss = ((X[:, ch] - Z @ beta) ** 2).sum()
        tss = ((X[:, ch] - X[:, ch].mean()) ** 2).sum()
        p[ch] = stats.f.sf(((tss - rss) / 2) / (rss / (len(X) - 3)), 2, len(X) - 3)
    return p


data = {m: premotor.monkey_table(sessions, m) for m in "JR"}
for m in "JR":
    t, X = data[m]
    p = significant_channels(t, X)
    print(f"{m}: {int((p < 0.05 / 96).sum())} of 96 channels significant (p < 0.05/96), "
          f"{int((p < 0.05).sum())} at p < 0.05 uncorrected; live channels {int((X.std(axis=0) > 0).sum())}; trials {len(t)}")
    tl, Xl = premotor.monkey_table(sessions, m, units=premotor.FILE_PMD_UNITS)
    print(f"   variant, units the file labels PMd (0-95): {int((significant_channels(tl, Xl) < 0.05 / 96).sum())} of 96")
    for d in sorted(t.date.unique()):
        sel = (t.date == d).values
        print(f"   {d}: {int((significant_channels(t[sel], X[sel]) < 0.05 / 96).sum())} of 96")

**Result: reproduced.** 72 of 96 channels for J and 80 of 96 for R regress significantly on
target x and y, against the paper's 74 and 80. Within single sessions the counts are lower (J 39
to 70, R 30 to 70). On the units the file labels PMd the counts are 74 (J) and 40 (R).

## pca-variance

> The top six PCs of the condition-averaged delay activity explain cumulatively 58, 82, 88, 92,
> 94 and 95% of the variance for J (120 conditions) and 81, 89, 92, 94, 95 and 96% for R (132
> conditions).

In [ ]:
PAPER_CUM = {"J": [58, 82, 88, 92, 94, 95], "R": [81, 89, 92, 94, 95, 96]}
planes = {}
for m in "JR":
    t, X = data[m]
    planes[m] = premotor.fit_plane(t, X, premotor.N_PCS[m])
    cum = 100 * np.cumsum(planes[m]["explained"])[:6]
    within = np.abs(cum - PAPER_CUM[m]) <= 0.1 * np.array(PAPER_CUM[m])
    print(f"{m}: {len(planes[m]['conditions'])} conditions; cumulative variance of PCs 1 to 6 (%): {np.round(cum, 1).tolist()}")
    print(f"   paper: {PAPER_CUM[m]}; within 10%: {within.tolist()}; outside by {np.round(np.maximum(np.abs(cum - PAPER_CUM[m]) - 0.1 * np.array(PAPER_CUM[m]), 0), 2).tolist()}")
    variants = {
        "day mean over the analysed delay windows": premotor.monkey_table(sessions, m, day_mean="trials"),
        "units the file labels PMd": premotor.monkey_table(sessions, m, units=premotor.FILE_PMD_UNITS),
        "no day-mean subtraction": (t, X + np.concatenate(
            [s["units"].rate.values[premotor.PMD_UNITS][None, :].repeat((s["trials"].analysed).sum(), 0)
             for s in sessions if s["monkey"] == m])),
    }
    for name, (tv, Xv) in variants.items():
        pv = premotor.fit_plane(tv, Xv, premotor.N_PCS[m])
        print(f"   variant, {name}: {np.round(100 * np.cumsum(pv['explained'])[:6], 1).tolist()}")

**Result: not reproduced.** R has 132 conditions, as in the paper. For J (96 of its 120
conditions) all six cumulative values are within tolerance (59.8 to 93.0% against 58 to 95%).
For R, PCs 2 to 6 are within tolerance (85.5 to 94.7% against 89 to 96%) and PC1 is 72.8% against
81 +- 8.1%, outside by 0.14 points. The criterion's partial case needs all of R's values within
tolerance, so by its letter the result is not reproduced, though it misses by a small margin.
The result depends on how the day's mean is taken: with the mean over the analysed delay
windows R's PC1 is 80.5%, and without day-mean subtraction 80.6%. The main choice, the rate over
the session's recorded time, is the rate over trial time, as the data check shows that spikes
exist only within trials.

## spatial-plane-variance

> The spatial plane captures a large part of the variance of the condition-averaged activity for
> the ring, horizontal, vertical and 3-rings tasks; 66.8, 64.4, 47.7 and 68.2% for J and 33.7,
> 30.7, 31.0 and 30.1% for R (p < 1e-4).

In [ ]:
PAPER_PLANE = {"J": {"ring": 66.8, "horizontal": 64.4, "vertical": 47.7, "3-rings": 68.2},
               "R": {"ring": 33.7, "horizontal": 30.7, "vertical": 31.0, "3-rings": 30.1}}
PLANE_SEED = 3


def task_means(t, X, task_column):
    """Per task, the condition means centred within the task."""
    out = {}
    for task in sorted(t[task_column].unique()):
        sel = (t[task_column] == task).values
        M = pd.DataFrame(X[sel]).groupby(t.condition.values[sel]).mean().values
        out[task] = M - M.mean(axis=0)
    return out


def plane_shares(means, plane):
    """Per task, share of the variance of its condition means in the plane: (a) of all channels,
    (b) of the top PCs."""
    Q = premotor.plane_basis(plane)
    return {task: {"a": 100 * ((Mc @ Q) ** 2).sum() / (Mc ** 2).sum(),
                   "b": 100 * ((Mc @ Q) ** 2).sum() / ((Mc @ plane["A"].T) ** 2).sum()}
            for task, Mc in means.items()}


def refit_with_targets(X, plane, tx, ty):
    """The plane refit from the same PCs to other target positions."""
    S = (X - plane["mu"]) @ plane["A"].T
    ones = np.ones((len(S), 1))
    bx = np.linalg.lstsq(np.hstack([ones, S]), tx, rcond=None)[0]
    null = np.eye(S.shape[1]) - np.outer(bx[1:], bx[1:]) / (bx[1:] @ bx[1:])
    by = np.linalg.lstsq(np.hstack([ones, S @ null]), ty, rcond=None)[0]
    return {**plane, "wx": plane["A"].T @ bx[1:], "wy": plane["A"].T @ (null @ by[1:])}


plane_results = {}
for m in "JR":
    t, X = data[m]
    codes, names = pd.factorize(t.condition)
    targets = t.groupby("condition")[["tx", "ty"]].first().loc[names].values
    rng = np.random.default_rng(PLANE_SEED)
    shuffled_planes = []
    for _ in range(1000):
        txy = targets[rng.permutation(len(targets))][codes]
        shuffled_planes.append(refit_with_targets(X, planes[m], txy[:, 0], txy[:, 1]))
    for task_column in ("methods_config", "config"):
        means = task_means(t, X, task_column)
        shares = plane_shares(means, planes[m])
        null = pd.DataFrame([{k: v["a"] for k, v in plane_shares(means, sp).items()} for sp in shuffled_planes])
        rows = []
        for task, v in shares.items():
            nul = null[task].values
            rows.append({"task": task, "(a) of all channels %": round(v["a"], 1), "(b) of top PCs %": round(v["b"], 1),
                         "paper %": PAPER_PLANE[m].get(task), "shuffle null (a) median %": round(np.median(nul), 1),
                         "shuffle null (a) 99th pct %": round(np.quantile(nul, 0.99), 1),
                         "p": (1 + np.sum(nul >= v["a"])) / (1 + len(nul))})
        res = pd.DataFrame(rows)
        plane_results[(m, task_column)] = res
        print(f"{m}, tasks by the {'Methods dates' if task_column == 'methods_config' else 'file geometry'}:")
        print(res.to_string(index=False))
    print(f"   smallest p attainable with {len(shuffled_planes)} shuffles: {1 / (1 + len(shuffled_planes)):.1e}")
    tv, Xv = premotor.monkey_table(sessions, m, day_mean="trials")
    pv = premotor.fit_plane(tv, Xv, premotor.N_PCS[m])
    print("   variant, day mean over the analysed delay windows, (a):",
          {k: float(round(v["a"], 1)) for k, v in plane_shares(task_means(tv, Xv, "methods_config"), pv).items()})

**Result: not reproduced.** No pairing of readings puts more than three of the seven values
within tolerance. For R the shares lie between 29.5 and 36.9% (reading a) against the paper's
30.1 to 33.7%, with ring at 29.9% against 33.7 +- 3.37%; for J (5 of 16 sessions, plane fit on 96
conditions) they are 54.4% (ring), 40.2% (vertical) and 44.3% (3-rings) against 66.8, 47.7 and
68.2%. Planes fit to targets shuffled across conditions capture median shares of 11.5 to 24.8%,
and the observed shares exceed the shuffles with p from 0.005 to 0.16. The smallest p that 1000
shuffles can give is 1.0e-03, but the observed p values lie well above it, so the paper's p <
1e-4 does not hold against this null; the paper does not say what its null was.

## plane-timecourse

> After target onset the neural state in the spatial plane starts at the center, diverges after
> 50 ms (R 75 ms) and converges to the steady delay state after 300 ms (R 400 ms).

In [ ]:
centers = premotor.TC_START_S + premotor.BIN_S * (np.arange(45) + 0.5)
bin_end = centers + premotor.BIN_S / 2
timecourse = {}
for m in "JR":
    t, X = data[m]
    plane = planes[m]
    coords, valid = [], []
    for s in sessions:
        if s["monkey"] != m:
            continue
        a = s["trials"][s["trials"].analysed]
        rates = s["tc_counts"] / premotor.BIN_S - s["units"].rate.values[premotor.PMD_UNITS][None, :, None]
        kernel = np.ones(3) / 3
        rates = np.apply_along_axis(lambda v: np.convolve(v, kernel, mode="same"), 2, rates)
        xy = premotor.plane_coords(plane, np.transpose(rates, (0, 2, 1)))  # trials x bins x 2
        coords.append(xy)
        # a bin is used while the go cue comes after it, including the bin the smoothing reaches
        valid.append(a.delay.values[:, None] >= bin_end[None, :] + premotor.BIN_S)
    coords, valid = np.concatenate(coords), np.concatenate(valid)
    conv = pd.DataFrame(premotor.plane_coords(plane, X)).groupby(t.condition.values).mean()
    d = np.full(len(centers), np.nan)
    n_cond = np.zeros(len(centers), int)
    for b in range(len(centers)):
        per = []
        for c, idx in t.groupby("condition").indices.items():
            ok = idx[valid[idx, b]]
            if len(ok) >= 5:
                per.append(np.linalg.norm(coords[ok, b].mean(axis=0) - conv.loc[c].values))
        if per:
            d[b], n_cond[b] = np.mean(per), len(per)
    d0 = np.nanmean(d[(centers > -0.1) & (centers < 0)])
    after = centers > 0
    dmin = np.nanmin(d[after])
    D = (d - dmin) / (d0 - dmin)
    diverge = centers[after][np.argmax(D[after] < 0.9)]
    converge = centers[after][np.argmax(D[after] < 0.1)]
    timecourse[m] = (d, D, n_cond)
    print(f"{m}: d0 {d0:.1f} mm, dmin {dmin:.1f} mm; divergence (D < 0.9) at {1000 * diverge:.0f} ms, "
          f"convergence (D < 0.1) at {1000 * converge:.0f} ms after target onset")
    for lo, hi in ((0.8, 0.1), (0.95, 0.2), (0.9, 0.05)):
        print(f"   thresholds {lo} and {hi}: {1000 * centers[after][np.argmax(D[after] < lo)]:.0f} and "
              f"{1000 * centers[after][np.argmax(D[after] < hi)]:.0f} ms")
    print("   D(t) every 40 ms from -90 ms:", {int(round(1000 * c)): float(round(v, 2)) for c, v in zip(centers[5::2], D[5::2])})
    print(f"   conditions with at least 5 trials per bin: {n_cond.min()} to {n_cond.max()}")
    pre = centers < 0
    print(f"   D over the bins before target onset (-200 to 0 ms): mean {np.nanmean(D[pre]):.2f}, SD {np.nanstd(D[pre]):.2f}, "
          f"range {np.nanmin(D[pre]):.2f} to {np.nanmax(D[pre]):.2f}; pre-onset bins below 0.9: {int(np.sum(D[pre] < 0.9))} of {pre.sum()}")
    # variant: at each bin, only trials whose 200 ms delay window starts after the bin (and its smoothing),
    # with the converged state taken from those same trials
    delays = np.concatenate([s["trials"][s["trials"].analysed].delay.values for s in sessions if s["monkey"] == m])
    P_delay = premotor.plane_coords(plane, X)
    d2 = np.full(len(centers), np.nan)
    for b in range(len(centers)):
        late = delays - premotor.DELAY_WINDOW_S >= bin_end[b] + premotor.BIN_S
        per = []
        for c, idx in t.groupby("condition").indices.items():
            ok = idx[late[idx]]
            if len(ok) >= 5:
                per.append(np.linalg.norm(coords[ok, b].mean(axis=0) - P_delay[ok].mean(axis=0)))
        if len(per) >= 0.5 * len(conv):
            d2[b] = np.mean(per)
    d20 = np.nanmean(d2[(centers > -0.1) & (centers < 0)])
    d2min = np.nanmin(d2[after])
    D2 = (d2 - d2min) / (d20 - d2min)
    last = centers[np.isfinite(d2)].max()
    hit = lambda thr: (f"{1000 * centers[after][np.nanargmax(D2[after] < thr)]:.0f} ms" if np.any(D2[after] < thr) else "never")
    print(f"   variant, converged state from trials whose delay window starts after the bin (bins up to {1000 * last:.0f} ms "
          f"with at least half the conditions): divergence {hit(0.9)}, convergence {hit(0.1)}; minimum at "
          f"{1000 * centers[after][np.nanargmin(D2[after])]:.0f} ms; pre-onset D SD {np.nanstd(D2[pre]):.2f}")
    origin = coords[:, centers < 0][valid[:, centers < 0]]
    print(f"   state before target onset, mean over trials (mm): x {origin[:, 0].mean():.1f}, y {origin[:, 1].mean():.1f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for i, m in enumerate("JR"):
    ax[i].plot(1000 * centers, timecourse[m][0], color="tab:blue")
    ax[i].axvline(0, color="grey", lw=0.5)
    ax[i].set(title=f"monkey {m}", xlabel="time from target onset (ms)", ylabel="distance from delay state (mm)")
plt.tight_layout()
plt.show()
print("Plotted ranges (mm):", {m: (float(round(np.nanmin(timecourse[m][0]), 1)), float(round(np.nanmax(timecourse[m][0]), 1))) for m in "JR"})

**Result: partially reproduced.** The distance from the delay state starts to fall (D below 0.9)
70 ms after target onset for J and 90 ms for R, against 50 and 75 ms, and reaches the delay state
(D below 0.1) at 270 ms for J and 350 ms for R, against 300 and 400 ms. J's times are robust: its
D varies little before target onset (SD 0.03, range 0.96 to 1.03), and with the converged state
taken only from trials whose delay window starts after the bin its times are 70 and 250 ms. R's
divergence time is not: its D lies below 0.9 in 6 of the 10 bins before target onset (range 0.87
to 1.11, SD 0.09), so a fall below 0.9 cannot be told from baseline noise, and with the
non-overlapping converged state R diverges at 10 ms, outside 75 +- 50 ms (convergence 310 ms).
The times of one monkey hold, so the result is partially reproduced. With other thresholds the
divergence moves between 10 and 150 ms and the convergence between 230 and 390 ms. Before target
onset the state sits at x -2.1, y -30.4 mm (J) and 17.6, -13.1 mm (R) of the plane's target
coordinates, so the statement that it starts at the center is not borne out.

## onoff-predicts-kinematics

> On single trials, on-axis and off-axis deviations of the neural state in the spatial plane are
> positively correlated with the on-axis and off-axis deviations of the hand position at maximum
> speed (on-axis p = 1e-3 J, 4e-3 R; off-axis p = 4e-4 J, 2e-4 R).

In [ ]:
devs = {}
for m in "JR":
    t, X = data[m]
    P = premotor.plane_coords(planes[m], X)
    ok = t.max_speed.notna().values
    tt = t[ok].reset_index(drop=True)
    nd = premotor.deviations(P[ok], tt.condition.values)
    kd = premotor.deviations(tt[["x_at_max", "y_at_max"]].values, tt.condition.values)
    n_on, n_off = premotor.on_off(nd, tt.tx.values, tt.ty.values)
    k_on, k_off = premotor.on_off(kd, tt.tx.values, tt.ty.values)
    devs[m] = (tt, P[ok], n_on, n_off)
    print(f"{m}: {len(tt)} trials ({int((~ok).sum())} without kinematics)")
    for name, sel in (("(a) all tasks", np.ones(len(tt), bool)), ("(b) 3-rings", (tt.config == "3-rings").values)):
        r1, p1 = stats.pearsonr(n_on[sel], k_on[sel])
        r2, p2 = stats.pearsonr(n_off[sel], k_off[sel])
        print(f"   {name}: on-axis r = {r1:.3f}, p = {p1:.1e}; off-axis r = {r2:.3f}, p = {p2:.1e} (n = {sel.sum()})")
    for d in sorted(tt.date.unique()):
        sel = (tt.date == d).values
        r1, p1 = stats.pearsonr(n_on[sel], k_on[sel])
        r2, p2 = stats.pearsonr(n_off[sel], k_off[sel])
        print(f"   {d}: on r = {r1:.3f} (p = {p1:.1e}), off r = {r2:.3f} (p = {p2:.1e})")
    sc = (tt.date + "|" + tt.condition).values
    nd2 = premotor.deviations(P[ok], sc)
    kd2 = premotor.deviations(tt[["x_at_max", "y_at_max"]].values, sc)
    n_on2, n_off2 = premotor.on_off(nd2, tt.tx.values, tt.ty.values)
    k_on2, k_off2 = premotor.on_off(kd2, tt.tx.values, tt.ty.values)
    for name, sel in (("(a) all tasks", np.ones(len(tt), bool)), ("(b) 3-rings", (tt.config == "3-rings").values)):
        r1, p1 = stats.pearsonr(n_on2[sel], k_on2[sel])
        r2, p2 = stats.pearsonr(n_off2[sel], k_off2[sel])
        print(f"   variant, deviations from session x condition means, {name}: on-axis r = {r1:.3f}, p = {p1:.1e}; "
              f"off-axis r = {r2:.3f}, p = {p2:.1e}")
    rs1, ps1 = stats.spearmanr(n_on, k_on)
    rs2, ps2 = stats.spearmanr(n_off, k_off)
    print(f"   variant, Spearman over all tasks: on {rs1:.3f} (p = {ps1:.1e}), off {rs2:.3f} (p = {ps2:.1e})")

**Result: reproduced.** Over all tasks, neural and kinematic deviations correlate positively on
axis (J r = 0.091, p = 1.7e-17; R r = 0.022, p = 1.3e-02) and off axis (J r = 0.102, p =
1.3e-21; R r = 0.072, p = 2.2e-16). With deviations from session by condition means, so that
offsets between sessions cannot correlate, the values are nearly the same (on axis J r = 0.092,
p = 9.7e-18, R r = 0.020, p = 2.3e-02; off axis J 0.101, R 0.083, p at most 3.0e-21). R's on-axis
correlation is weak: in the 3-rings task alone it is not significant (r = 0.051, p = 6.5e-02),
and within sessions it is positive with p < 0.05 only on 2017-01-15 and negative on 2017-01-16
(r = -0.044, p = 4.9e-02).

## speed-axis-within-distance

> The speed axis of delay activity varies with maximum speed for reaches to the same target
> distance, for 11 of 12 distances in J and 11 of 12 in R (p < 0.05), so it encodes speed
> independently of distance.

In [ ]:
SPEED_SEED = 4


def speed_axis(Xc, speed, Q):
    """Regression of maximum speed on the activity with the plane projected out (unit-free axis)."""
    Xp = Xc - (Xc @ Q) @ Q.T
    fit = LinearRegression().fit(Xp, speed)
    return fit, Xp


speed_fits = {}
for m in "JR":
    t, X = data[m]
    ok = t.max_speed.notna().values
    tt, Xo = t[ok].reset_index(drop=True), X[ok]
    Xc = Xo - Xo.mean(axis=0)
    Q = premotor.plane_basis(planes[m])
    lines = (tt.config.isin(["horizontal", "vertical"])).values
    # the k-th distance of each line task (R's horizontal targets sit at 5.2 + 10.4 k mm)
    tt["distance_rank"] = tt.groupby("config").distance.rank(method="dense").astype(int)
    print(f"{m}: {len(tt)} trials with kinematics, {lines.sum()} horizontal or vertical "
          f"({sorted(tt[lines].config.unique())}); distances (mm) by task "
          f"{ {c: sorted(np.round(g.distance, 1).unique().tolist()) for c, g in tt[lines].groupby('config')} }")
    for reading, fit_rows in (("(a) fit on all tasks", np.ones(len(tt), bool)), ("(b) fit on horizontal and vertical", lines)):
        fit, Xp = speed_axis(Xc[fit_rows], tt.max_speed.values[fit_rows], Q)
        proj = (Xc - (Xc @ Q) @ Q.T) @ fit.coef_
        if reading.startswith("(a)"):
            speed_fits[m] = (tt, Xc, fit.coef_)
        # cross-validated projections (variant)
        oof = np.full(len(tt), np.nan)
        idx = np.flatnonzero(fit_rows)
        for train, test in KFold(10, shuffle=True, random_state=SPEED_SEED).split(idx):
            f2, _ = speed_axis(Xc[idx[train]], tt.max_speed.values[idx[train]], Q)
            oof[idx[test]] = (Xc[idx[test]] - (Xc[idx[test]] @ Q) @ Q.T) @ f2.coef_
        out = []
        for dist in range(1, 13):
            sel = lines & (tt.distance_rank == dist).values
            r, p = stats.pearsonr(proj[sel], tt.max_speed[sel])
            rc, pc = stats.pearsonr(oof[sel], tt.max_speed[sel]) if np.isfinite(oof[sel]).all() else (np.nan, np.nan)
            out.append({"distance_rank": dist, "n": int(sel.sum()), "r": round(r, 3), "p": p,
                        "r_cv": round(rc, 3), "p_cv": pc})
        out = pd.DataFrame(out)
        n_sig = int(((out.r > 0) & (out.p < 0.05)).sum())
        n_sig_cv = int(((out.r_cv > 0) & (out.p_cv < 0.05)).sum())
        print(f"   {reading}: significant positive at {n_sig} of {len(out)} distances; cross-validated {n_sig_cv} of {len(out)}")
        print(out.to_string(index=False, formatters={"p": "{:.1e}".format, "p_cv": "{:.1e}".format}))
    proj_a = (Xc - (Xc @ Q) @ Q.T) @ speed_fits[m][2]
    split = []
    for (task, date), g in tt[lines].groupby(["config", "date"]):
        n_pos = 0
        for dist in range(1, 13):
            idx = g.index[g.distance_rank == dist]
            r, p = stats.pearsonr(proj_a[idx], tt.max_speed[idx])
            n_pos += (r > 0) & (p < 0.05)
        split.append(f"{task} {date}: {n_pos} of 12")
    print("   reading (a), significant positive distances within each task and session:", "; ".join(split))
    r_all, p_all = stats.pearsonr(tt.max_speed, tt.distance)
    print(f"   maximum speed against target distance over all trials: r = {r_all:.3f}, p = {p_all:.1e}")

**Result: partially reproduced.** For R the speed axis correlates positively with maximum speed
at 11 of 12 distances under both readings, as in the paper, and also with cross-validated
projections; the exception is the nearest distance. For J, whose horizontal sessions are not in
the dandiset, it does at 6 of 12 distances under both readings (4 and 5 cross-validated), on 3211
vertical trials. Each distance rank pools both line tasks and both sessions of each; R's
horizontal (5.2 + 10.4 k mm) and vertical (10 k mm) targets of the same rank differ by up to 5
mm, and rank 12 (both at 120 mm) is significant. Split by task and session, 110 to 180 trials per distance, R has 1, 4, 10 and 7 of 12 (horizontal 2017-01-15 and 16, vertical 01-17 and 18) and J
1 and 3, so the counts rest on pooling sessions.

## distance-after-speed

> After regressing out speed from the neural activity, a positive correlation with target
> distance remains (p < 1e-4 for J and R).

In [ ]:
DIST_SEED = 5
distance_axes = {}
for m in "JR":
    tt, Xc, ws = speed_fits[m]
    s = tt.max_speed.values - tt.max_speed.mean()
    Xr = Xc - np.outer(s, Xc.T @ s / (s @ s))
    y = tt.distance.values
    oof = np.empty(len(y))
    for train, test in KFold(10, shuffle=True, random_state=DIST_SEED).split(Xr):
        oof[test] = LinearRegression().fit(Xr[train], y[train]).predict(Xr[test])
    r, p = stats.pearsonr(oof, y)
    full = LinearRegression().fit(Xr, y)
    ri, pi = stats.pearsonr(full.predict(Xr), y)
    distance_axes[m] = full.coef_
    print(f"{m}: cross-validated prediction of target distance from speed-residualised activity: r = {r:.3f}, p = {p:.1e} "
          f"(n = {len(y)}); in-sample r = {ri:.3f}, p = {pi:.1e}")
    print(f"   residual correlation of every channel with speed (max |r|): "
          f"{np.nanmax(np.abs([stats.pearsonr(Xr[:, c], s)[0] if Xr[:, c].std() > 0 else 0 for c in range(Xr.shape[1])])):.1e}")
    for c in sorted(tt.config.unique()):
        sel = (tt.config == c).values
        if np.ptp(np.round(y[sel])) == 0:
            continue  # one distance (ring): no correlation to test
        print(f"   {c}: cross-validated r = {stats.pearsonr(oof[sel], y[sel])[0]:.3f}, p = {stats.pearsonr(oof[sel], y[sel])[1]:.1e}")

**Result: reproduced.** With speed regressed out, cross-validated predictions of target distance
correlate with distance (J r = 0.123, p = 1.2e-30; R r = 0.233, p = 7.1e-158). Within the tasks
whose targets lie at several distances the correlation holds too: 3-rings J r = 0.154, R r =
0.265; vertical J 0.049 (p = 5.6e-03), R 0.214; horizontal R 0.355. The ring task, with one
distance, is not tested.

## speed-distance-projection

> Projected onto the top PCs, the distance dimension is about half as large as the speed
> dimension.

In [ ]:
for m in "JR":
    A = planes[m]["A"]
    _, _, ws = speed_fits[m]
    wd = distance_axes[m]
    ps, pd_ = np.linalg.norm(A @ (ws / np.linalg.norm(ws))), np.linalg.norm(A @ (wd / np.linalg.norm(wd)))
    print(f"{m}: norm of the projection onto the top {A.shape[0]} PCs, speed axis {ps:.3f}, distance axis {pd_:.3f}; "
          f"ratio {pd_ / ps:.2f}")
    Q = premotor.plane_basis(planes[m])
    wd_perp = wd - Q @ (Q.T @ wd)
    print(f"   variant, distance axis with the plane projected out: projection {np.linalg.norm(A @ (wd_perp / np.linalg.norm(wd_perp))):.3f}, "
          f"ratio {np.linalg.norm(A @ (wd_perp / np.linalg.norm(wd_perp))) / ps:.2f}")
    print(f"   cosine between speed and distance axes: {ws @ wd / np.linalg.norm(ws) / np.linalg.norm(wd):.3f}")

**Result: not reproduced.** Projected onto the top PCs, the distance axis is as large as the
speed axis for J (ratio 1.01) and larger for R (1.68), not about half. With the plane projected
out of the distance axis the ratios are 0.73 and 1.14.

## single-trial-variance

> Across all single trials the speed dimension explains 2% (J) and 1% (R) of the variance of the
> neural state, and the spatial plane 20% (J) and 10% (R).

In [ ]:
for m in "JR":
    t, X = data[m]
    Xc = X - X.mean(axis=0)
    total = (Xc ** 2).sum()
    _, _, ws = speed_fits[m]
    u = ws / np.linalg.norm(ws)
    Q = premotor.plane_basis(planes[m])
    print(f"{m}: speed axis {100 * ((Xc @ u) ** 2).sum() / total:.2f}% of the single-trial variance; spatial plane "
          f"{100 * ((Xc @ Q) ** 2).sum() / total:.1f}%; top {premotor.N_PCS[m]} PCs "
          f"{100 * ((Xc @ planes[m]['A'].T) ** 2).sum() / total:.1f}% (trials {len(Xc)})")
    tv, Xv = premotor.monkey_table(sessions, m, day_mean="trials")
    Xvc = Xv - Xv.mean(axis=0)
    pv = premotor.fit_plane(tv, Xv, premotor.N_PCS[m])
    Qv = premotor.plane_basis(pv)
    print(f"   variant, day mean over the analysed delay windows: plane {100 * ((Xvc @ Qv) ** 2).sum() / (Xvc ** 2).sum():.1f}%, "
          f"speed axis (same direction) {100 * ((Xvc @ u) ** 2).sum() / (Xvc ** 2).sum():.2f}%")
    per = []
    for d in sorted(t.date.unique()):
        sel = (t.date == d).values
        Xs = X[sel] - X[sel].mean(axis=0)
        per.append(f"{d} {100 * ((Xs @ Q) ** 2).sum() / (Xs ** 2).sum():.1f}%")
    print("   plane share within each session:", "; ".join(per))

**Result: not reproduced.** The speed axis explains 1.25% of the single-trial variance for J
and 0.86% for R, against 2% and 1%; the plane explains 15.7% and 8.0%, against 20% and 10%. Only
R's speed value is within tolerance. The values are lower than the paper's in all four cases and
keep its order (J above R).

## ellipse-orientation

> The distributions of single-trial neural states for each condition are elliptical, and for
> most conditions their major axis points along the direction of the mean neural state from the
> origin.

In [ ]:
for m in "JR":
    t, X = data[m]
    P = premotor.plane_coords(planes[m], X)
    rows = []
    for c, idx in t.groupby("condition").indices.items():
        if len(idx) < 10:
            continue
        pts = P[idx]
        mean = pts.mean(axis=0)
        evals, evecs = np.linalg.eigh(np.cov(pts.T))
        major = evecs[:, -1]
        angle = np.degrees(np.arccos(min(1.0, abs(major @ mean) / np.linalg.norm(mean))))
        rows.append({"condition": c, "config": t.config.iloc[idx[0]], "angle": angle,
                     "axis_ratio": np.sqrt(evals[-1] / evals[0]), "mean_radius": np.linalg.norm(mean)})
    e = pd.DataFrame(rows)
    w = stats.wilcoxon(e.angle - 45, alternative="less")
    print(f"{m}: {len(e)} conditions; angle below 45 degrees in {int((e.angle < 45).sum())} ({np.mean(e.angle < 45):.2f}); "
          f"median angle {e.angle.median():.1f} degrees; Wilcoxon against 45, p = {w.pvalue:.1e}; "
          f"median ratio of major to minor axis {e.axis_ratio.median():.2f}")
    print("   by configuration:")
    print(e.groupby("config").angle.agg(conditions="count", median_angle="median",
                                         below_45=lambda a: np.mean(a < 45)).round(2).to_string())
    near = e.mean_radius < e.mean_radius.median()
    print(f"   conditions with the condition mean nearer the origin than the median: {np.mean(e.angle[near] < 45):.2f} below 45; "
          f"farther: {np.mean(e.angle[~near] < 45):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4.3))
for i, m in enumerate("JR"):
    t, X = data[m]
    P = premotor.plane_coords(planes[m], X)
    sel = (t.config == "3-rings").values
    for c, idx in t[sel].groupby("condition").indices.items():
        g = P[np.flatnonzero(sel)[idx]]
        mean = g.mean(axis=0)
        evals, evecs = np.linalg.eigh(np.cov(g.T))
        theta = np.linspace(0, 2 * np.pi, 60)
        ell = mean + (evecs @ (np.sqrt(evals)[:, None] * np.stack([np.cos(theta), np.sin(theta)]))).T / 3
        ax[i].plot(ell[:, 0], ell[:, 1], lw=0.8)
    ax[i].set(title=f"monkey {m}, 3-rings, 1/3 SD ellipses", xlabel="x neural (mm)", ylabel="y neural (mm)", aspect="equal")
plt.tight_layout()
plt.show()

**Result: reproduced.** The major axis lies within 45 degrees of the direction of the condition
mean for 69 of 96 conditions in J (median 21.0 degrees, Wilcoxon p = 4.1e-08) and 99 of 132 in R
(median 19.3 degrees, p = 8.0e-10). The median ratio of major to minor axis is 1.35 (J) and
1.56 (R).

## on-greater-than-off

> The mean absolute on-axis (distance) deviation of the neural state is larger than the off-axis
> (direction) deviation, in each task (J p = 9e-7 ring, 4e-7 horizontal, 1e-8 vertical, 3e-14
> 3-rings; R 8e-3, 4e-7, 2e-7 and 0.19).

In [ ]:
PAPER_P = {"J": {"ring": 9e-7, "horizontal": 4e-7, "vertical": 1e-8, "3-rings": 3e-14},
           "R": {"ring": 8e-3, "horizontal": 4e-7, "vertical": 2e-7, "3-rings": 0.19}}
onoff_rows = []
for m in "JR":
    t, X = data[m]
    P = premotor.plane_coords(planes[m], X)
    dev = premotor.deviations(P, t.condition.values)
    on, off = premotor.on_off(dev, t.tx.values, t.ty.values)
    t = t.assign(on=np.abs(on), off=np.abs(off))
    for task_column in ("methods_config", "config"):
        for task in sorted(t[task_column].unique()):
            g = t[t[task_column] == task]
            per = g.groupby("condition")[["on", "off"]].mean()
            pa = stats.ttest_rel(per.on, per.off).pvalue
            pb = stats.ttest_ind(per.on, per.off, equal_var=False).pvalue
            pc = stats.ttest_ind(g.on, g.off, equal_var=False).pvalue
            onoff_rows.append({"monkey": m, "tasks by": "Methods" if task_column == "methods_config" else "file",
                               "task": task, "targets": len(per), "mean |on| mm": round(per.on.mean(), 2),
                               "mean |off| mm": round(per.off.mean(), 2), "targets on > off": int((per.on > per.off).sum()),
                               "(a) paired p": pa, "(b) Welch targets p": pb, "(c) Welch trials p": pc,
                               "paper p": PAPER_P[m][task]})
onoff = pd.DataFrame(onoff_rows)
print(onoff.to_string(index=False, formatters={k: "{:.1e}".format for k in ["(a) paired p", "(b) Welch targets p", "(c) Welch trials p", "paper p"]}))
main = onoff[(onoff["tasks by"] == "Methods") & (onoff["paper p"] < 0.05)]
for r in ["(a) paired p", "(b) Welch targets p", "(c) Welch trials p"]:
    ok = (main["mean |on| mm"] > main["mean |off| mm"]) & (main[r] < 0.05)
    print(f"{r}: on > off with p < 0.05 in {int(ok.sum())} of {len(main)} task and monkey pairs the paper finds significant")

**Result: partially reproduced.** On-axis deviations exceed off-axis ones with p < 0.05 in 5 of
the 6 task and monkey pairs under readings (a) and (c), and 4 of 6 under (b). The exception is
J's ring task, which here is one session of four (13 of 24 targets with on > off, paired p =
0.69). R's 3-rings task shows no significant difference (paired p = 0.38), as in the paper.

## relative-deviation-decreases

> Relative deviations of the neural state (deviation as a fraction of reach distance), on and off
> axis, decrease with reach distance.

In [ ]:
for m in "JR":
    t, X = data[m]
    P = premotor.plane_coords(planes[m], X)
    on, off = premotor.on_off(premotor.deviations(P, t.condition.values), t.tx.values, t.ty.values)
    dist = t.distance.values
    for name, v in (("on-axis", on), ("off-axis", off)):
        rel = np.abs(v) / dist
        rho, p = stats.spearmanr(dist, rel)
        rho_abs, p_abs = stats.spearmanr(dist, np.abs(v))
        print(f"{m} {name}: Spearman of relative deviation with distance rho = {rho:.3f}, p = {p:.1e}; "
              f"absolute deviation rho = {rho_abs:.3f}, p = {p_abs:.1e}")
    binned = pd.DataFrame({"distance": np.round(dist / 10) * 10, "on": 100 * np.abs(on) / dist, "off": 100 * np.abs(off) / dist})
    print("   mean relative deviation (% of distance) by distance (mm):")
    print(binned.groupby("distance").mean().round(1).T.to_string())

**Result: reproduced.** Relative deviations fall with reach distance on axis (J rho = -0.389, R
rho = -0.404) and off axis (J -0.378, R -0.402), all with p below 1e-290. The absolute deviations
grow only slightly with distance (rho 0.073 to 0.103), so the fall comes mainly from dividing by
distance, and the test would pass for any deviation that grows less than in proportion to
distance; it discriminates little.

## svm-ring-and-distance

> Direction classification of the ring targets reaches 32.2% for J (24 targets) and 16% for R
> (36 targets); distance classification over 12 targets reaches 18.9% (J) and 21.4% (R).

In [ ]:
SVM_SEED = 6
svm_out = {}
for m in "JR":
    t, X = data[m]
    ring = (t.config == "ring").values
    y = np.round(np.degrees(np.arctan2(t.ty[ring], t.tx[ring]))).astype(int).values
    pred, acc = premotor.svm_cv(X[ring], y, SVM_SEED)
    print(f"{m} ring direction: {100 * acc.mean():.1f}% +- {100 * acc.std(ddof=1) / np.sqrt(len(acc)):.1f}% (SEM over 10 folds); "
          f"chance {100 / len(np.unique(y)):.1f}%; trials {ring.sum()} in {len(np.unique(y))} targets, "
          f"sessions {sorted(t[ring].date.unique())}")
    for d in sorted(t[ring].date.unique()):
        sel = ring & (t.date == d).values
        ys = np.round(np.degrees(np.arctan2(t.ty[sel], t.tx[sel]))).astype(int).values
        print(f"   {d} alone: {100 * premotor.svm_cv(X[sel], ys, SVM_SEED)[1].mean():.1f}% ({sel.sum()} trials)")
    rows, errors, errors_plane = [], [], []
    P = premotor.plane_coords(planes[m], X)
    for task in ("horizontal", "vertical"):
        for side in (1, -1):
            coord = t.tx if task == "horizontal" else t.ty
            sel = ((t.config == task) & (np.sign(coord) == side)).values
            if not sel.any():
                continue
            yd = np.round(t.distance[sel], 1).values
            pred, acc = premotor.svm_cv(X[sel], (10 * yd).astype(int), SVM_SEED)
            pred_p, acc_p = premotor.svm_cv(P[sel], (10 * yd).astype(int), SVM_SEED)
            errors.append(np.abs(pred / 10 - yd) / 10)        # cm
            errors_plane.append(np.abs(pred_p / 10 - yd) / 10)
            rows.append({"task": task, "side": side, "trials": int(sel.sum()), "accuracy %": 100 * acc.mean(),
                         "SEM %": 100 * acc.std(ddof=1) / np.sqrt(len(acc)), "plane-projected accuracy %": 100 * acc_p.mean()})
    dist = pd.DataFrame(rows)
    weighted = np.average(dist["accuracy %"], weights=dist.trials)
    print(f"{m} distance (12 targets per side; chance 8.3%):")
    print(dist.round(1).to_string(index=False))
    print(f"   accuracy weighted by trials: {weighted:.1f}%")
    svm_out[m] = {"distance_errors": np.concatenate(errors), "distance_errors_plane": np.concatenate(errors_plane)}
    Xl = premotor.monkey_table(sessions, m, units=premotor.FILE_PMD_UNITS)[1]
    print(f"   variant, units the file labels PMd: ring direction "
          f"{100 * premotor.svm_cv(Xl[ring], y, SVM_SEED)[1].mean():.1f}%")

Variant for this claim and svm-3rings-by-ring: hyperparameters chosen on the 3-rings task, as the
paper did (its held-out 3-rings trials are not identified, so here the same 3-rings trials are
used to choose them), then applied to the ring, distance and per-ring classifiers.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold  # noqa: E402
from sklearn.pipeline import make_pipeline  # noqa: E402
from sklearn.preprocessing import StandardScaler  # noqa: E402
from sklearn.svm import SVC  # noqa: E402

TUNE_SEED = 8
for m in "JR":
    t, X = data[m]
    sel = (t.config == "3-rings").values
    y = (np.round(np.degrees(np.arctan2(t.ty[sel], t.tx[sel]))) * 1000 + np.round(t.distance[sel])).astype(int).values
    grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                        {"svc__C": [0.3, 1, 3, 10], "svc__gamma": [0.1 / 96, 1 / 96, 3 / 96]},
                        cv=StratifiedKFold(3, shuffle=True, random_state=TUNE_SEED)).fit(X[sel], y)
    C, gamma = grid.best_params_["svc__C"], grid.best_params_["svc__gamma"]
    print(f"{m}: best on 3-rings (48 targets): C = {C}, gamma = {gamma * 96:.1f}/96, accuracy {100 * grid.best_score_:.1f}% "
          f"(default C = 1, gamma = 1/96 after scaling: "
          f"{100 * grid.cv_results_['mean_test_score'][list(grid.cv_results_['params']).index({'svc__C': 1, 'svc__gamma': 1 / 96})]:.1f}%)")

    def tuned(Xs, ys):
        accs = []
        for tr, te in StratifiedKFold(10, shuffle=True, random_state=SVM_SEED).split(Xs, ys):
            model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma)).fit(Xs[tr], ys[tr])
            accs.append(np.mean(model.predict(Xs[te]) == ys[te]))
        return 100 * np.mean(accs)

    ring = (t.config == "ring").values
    yr = np.round(np.degrees(np.arctan2(t.ty[ring], t.tx[ring]))).astype(int).values
    acc_d, n_d = [], []
    for task in ("horizontal", "vertical"):
        for side in (1, -1):
            coord = t.tx if task == "horizontal" else t.ty
            s_ = ((t.config == task) & (np.sign(coord) == side)).values
            if s_.any():
                acc_d.append(tuned(X[s_], (np.round(t.distance[s_], 1) * 10).astype(int).values))
                n_d.append(s_.sum())
    rings = [tuned(X[(t.config == "3-rings").values & (np.round(t.distance) == r).values],
                   np.round(np.degrees(np.arctan2(t.ty, t.tx)))[(t.config == "3-rings").values & (np.round(t.distance) == r).values].astype(int).values)
             for r in (40, 80, 120)]
    print(f"   with these: ring direction {tuned(X[ring], yr):.1f}%, distance {np.average(acc_d, weights=n_d):.1f}%, "
          f"3-rings by ring {[float(round(a, 1)) for a in rings]}%")

**Result: not reproduced.** All four accuracies are below the paper's: ring direction 24.7%
(J, one session of four) and 12.9% (R) against 32.2 and 16%, and distance 16.3% (J, vertical
only) and 17.1% (R) against 18.9 and 21.4%. All are well above chance (4.2, 2.8 and 8.3%). With
hyperparameters chosen on the 3-rings trials the values are 24.3, 13.8, 16.0 and 18.3%.

## svm-3rings-by-ring

> Classifying the 16 directions of each ring of the 3-rings task separately, accuracy increases
> with ring distance; 23, 36 and 43% for J and 22, 28 and 29% for R.

In [ ]:
RINGS_SEED = 7
PAPER_RINGS = {"J": [23, 36, 43], "R": [22, 28, 29]}
for m in "JR":
    t, X = data[m]
    accs, arc, ang = [], {}, {}
    for radius in (40, 80, 120):
        sel = ((t.config == "3-rings") & (np.round(t.distance) == radius)).values
        y = np.round(np.degrees(np.arctan2(t.ty[sel], t.tx[sel]))).astype(int).values
        pred, acc = premotor.svm_cv(X[sel], y, RINGS_SEED)
        err = np.abs((pred - y + 180) % 360 - 180)  # degrees
        ang[radius] = err
        arc[radius] = np.radians(err) * radius / 10  # cm
        accs.append(100 * acc.mean())
        print(f"{m} {radius // 10} cm ring: {100 * acc.mean():.1f}% +- {100 * acc.std(ddof=1) / np.sqrt(len(acc)):.1f}%, "
              f"chance {100 / len(np.unique(y)):.2f}%, trials {sel.sum()}; mean angular error {err.mean():.1f} deg, "
              f"mean arc error {arc[radius].mean():.2f} cm")
    print(f"   paper {PAPER_RINGS[m]}; within 10%: {[bool(abs(a - p) <= 0.1 * p) for a, p in zip(accs, PAPER_RINGS[m])]}; "
          f"increasing: {accs[0] < accs[1] < accs[2]}")
    svm_out[m].update(arc=arc, ang=ang)

**Result: partially reproduced.** Accuracy rises with ring distance in both monkeys. J's values,
23.4, 34.2 and 42.3%, are within tolerance of 23, 36 and 43%; R's, 18.4, 21.9 and 22.0%, are
below 22, 28 and 29%.

## arc-error-distance-independent

> Angular errors of direction classification decrease with ring distance, but errors measured as
> arc length are not significantly different between rings (F test, p > .05 for J and R).

In [ ]:
for m in "JR":
    arc, ang = svm_out[m]["arc"], svm_out[m]["ang"]
    F, p = stats.f_oneway(*arc.values())
    Fa, pa = stats.f_oneway(*ang.values())
    print(f"{m}: mean angular error 4/8/12 cm {[float(round(ang[r].mean(), 1)) for r in (40, 80, 120)]} deg (ANOVA p = {pa:.1e}); "
          f"mean arc error {[float(round(arc[r].mean(), 2)) for r in (40, 80, 120)]} cm; one-way ANOVA F = {F:.2f}, p = {p:.1e}")
    limit = np.pi * 4
    restricted = {r: a[a <= limit] for r, a in arc.items()}
    Fr, pr = stats.f_oneway(*restricted.values())
    print(f"   variant, errors within the 4 cm ring's range (<= {limit:.2f} cm): means "
          f"{[float(round(restricted[r].mean(), 2)) for r in (40, 80, 120)]} cm, F = {Fr:.2f}, p = {pr:.1e}; trials kept "
          f"{[len(restricted[r]) for r in (40, 80, 120)]} of {[len(arc[r]) for r in (40, 80, 120)]}")
    lev = stats.levene(*arc.values())
    print(f"   variant, Levene test of equal variances of arc errors: p = {lev.pvalue:.1e}; "
          f"arc error 12 cm minus 4 cm ring {arc[120].mean() - arc[40].mean():.2f} cm")

**Result: not reproduced.** Mean angular errors fall with ring distance (J 41.0, 24.3, 18.3
degrees; R 52.0, 42.4, 36.7), but arc-length errors grow with it (J 2.87, 3.40, 3.84 cm; R 3.63,
5.92, 7.69 cm), and the one-way ANOVA finds the rings different (J F = 22.15, p = 2.7e-10; R F =
52.16, p = 1.5e-22). Restricted to errors within the 4 cm ring's range, J's rings no longer
differ (p = 0.23) and R's still do (p = 1.3e-03).

## distance-error-and-plane-projection

> Mean distance-classification error is slightly but significantly lower than the arc-length
> error of direction classification, and projecting the activity onto the spatial plane before
> classification raises it (J 2.64 to 3.03 cm; R given as 2.78 and 3.27 cm), to the level of the
> direction errors.

In [ ]:
for m in "JR":
    full, plane_err = svm_out[m]["distance_errors"], svm_out[m]["distance_errors_plane"]
    w = stats.ttest_ind(plane_err, full, equal_var=False)
    pr = stats.ttest_rel(plane_err, full)
    print(f"{m}: distance error full {full.mean():.3f} cm, plane-projected {plane_err.mean():.3f} cm "
          f"(Welch t = {w.statistic:.2f}, p = {w.pvalue:.1e}; paired over trials t = {pr.statistic:.2f}, p = {pr.pvalue:.1e}; n = {len(full)})")
    for r in (40, 80, 120):
        a = svm_out[m]["arc"][r]
        print(f"   direction arc error, {r // 10} cm ring {a.mean():.2f} cm; full distance error lower: "
              f"p = {stats.ttest_ind(full, a, equal_var=False).pvalue:.1e}; plane-projected against arc: "
              f"p = {stats.ttest_ind(plane_err, a, equal_var=False).pvalue:.1e}")

**Result: reproduced.** Under reading (a), mean distance errors are 2.505 cm (J) and 2.503 cm
(R) with the full activity and 2.801 and 3.026 cm after projection onto the plane, all within
tolerance; R's full value is inside its bound (2.502 cm) by 0.001 cm, so this match is fragile.
Projection raises the error in both (Welch J p = 2.7e-07, R p = 4.3e-41; paired over trials J p
= 4.9e-12, R p = 1.2e-62). The plane is fit on all trials, including each test fold, which
favours the projected classifier, so the rise is if anything understated. The full-activity
distance error is lower than the direction arc errors of every ring in both monkeys (p at most
3.1e-05). After projection it is no different from the arc error of J's 4 cm ring (p = 0.46)
and differs from the others.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| session-configurations | partially reproduced |
| r-trial-counts | partially reproduced |
| j-sessions-complete | blocked (data missing) |
| delays-and-times | partially reproduced |
| channels-task-correlated | reproduced |
| pca-variance | not reproduced |
| spatial-plane-variance | not reproduced |
| plane-timecourse | partially reproduced |
| onoff-predicts-kinematics | reproduced |
| speed-axis-within-distance | partially reproduced |
| distance-after-speed | reproduced |
| speed-distance-projection | not reproduced |
| single-trial-variance | not reproduced |
| ellipse-orientation | reproduced |
| on-greater-than-off | partially reproduced |
| relative-deviation-decreases | reproduced |
| svm-ring-and-distance | not reproduced |
| svm-3rings-by-ring | partially reproduced |
| arc-error-distance-independent | not reproduced |
| distance-error-and-plane-projection | reproduced |

### How the analysis differs from the paper's

- Monkey J is represented by 5 of its 16 sessions and 96 of its 120 conditions; its PCA, plane and
  classifiers are fit on these alone. Monkey R's data are complete.
- The PMd array is taken as units 96 to 191, against the units' `electrode_group` labels, from the
  data check; results on the labelled array are shown as variants where they bear on a claim.
- The paper does not say how the day's average rate, movement onset, maximum speed, the SVM
  hyperparameters, the folds or the null of its p-values were set; the choices here are listed in
  the claims file and in `premotor.py`. The day's mean rate is the rate over trial time, as
  spikes exist only within trials.
- The spatial plane is fit on all trials, so the plane-projected distance classifier sees its
  test folds through the plane; the paper does not say how it handled this.
- R's horizontal and vertical tasks are named by the Methods' dates; the file geometry is shown as
  a variant. R's horizontal targets lie at 5.2 + 10.4 k mm, so its 12 distances are matched by
  rank.

### What was found about the dandiset

- The units' `electrode_group` labels are swapped: the array labelled PMd is the one whose units
  point to M1 electrode rows and which, in R, has 29 silent channels and no target signal.
- In every J file the boundary between units 0 and 1 is misplaced, so unit 0 stops early and unit
  1's spike times run past the session's end and restart.
- Positions are stated in meters with `conversion` NaN but are in mm; `task_id` is the same in
  every trial, so the configuration must be inferred from the targets; `is_successful` carries a
  description of something else; electrode positions and impedances are empty.
- The archive's summary gives 15 files and 187.5 GB for a version of 12 files and 39.5 GB.
- 11 of J's 16 sessions, including all its horizontal sessions, are not in the dandiset.
- The files place R's horizontal and vertical tasks on the dates the Methods give the other way
  round; R has almost no trials without a delay and delays up to 1000 ms; its 3-rings session holds
  1340 successful delayed trials against the 912 the paper used.
- Spikes and hand samples exist only within trials; the gaps between trials (484 to 1963 s per
  session) hold almost no data.